In [ ]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential


In [ ]:
data_df = pd.read_csv('Supplementary data S6- input_combined.csv')
data_df


In [ ]:
# Hyperparameter tuning (run separately; computationally expensive)
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.metrics import roc_auc_score
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense
# from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
# from sklearn.model_selection import GridSearchCV

# data_df = pd.read_csv('Supplementary data S6- input_combined.csv')

# X = data_df.drop(columns=['Activity'])
# y = data_df['Activity'].map({'Inactive': 0, 'Active': 1})

# X_train, X_val, y_train, y_val = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=42
# )
# metrics = [tf.keras.metrics.AUC()]

# def create_deep_model(num_hidden_layers, num_neurons):
#     model = Sequential()
#     model.add(Dense(881, activation='relu', input_shape=(X.shape[1],)))
#
#     if num_hidden_layers == 3:
#         for neurons in num_neurons:
#             model.add(Dense(neurons, activation='relu'))
#     elif num_hidden_layers == 2:
#         model.add(Dense(num_neurons[0], activation='relu'))
#     else:
#         raise ValueError("Invalid number of hidden layers")
#
#     model.add(Dense(1, activation='sigmoid'))
#     model.compile(
#         optimizer='adam',
#         loss='binary_crossentropy',
#         metrics=metrics
#     )
#     return model

# param_grid = {
#     'num_hidden_layers': [3, 2],
#     'num_neurons': [
#         (512, 256, 128),
#         (256, 128, 64),
#         (128, 128, 64),
#         (512, 256),
#         (256, 128),
#         (128, 64)
#     ],
#     'epochs': [50, 100],
#     'batch_size': [32, 64]
# }

# model = KerasClassifier(build_fn=create_deep_model, verbose=0)
# cv = StratifiedKFold(n_splits=3)
# grid = GridSearchCV(
#     estimator=model,
#     param_grid=param_grid,
#     scoring='roc_auc',
#     cv=cv
# )
# grid_result = grid.fit(X_train, y_train)

# print("Best Parameters:")
# print("Number of Hidden Layers:", grid_result.best_params_['num_hidden_layers'])
# print("Number of Neurons in Hidden Layers:", grid_result.best_params_['num_neurons'])
# print("Number of Epochs:", grid_result.best_params_['epochs'])
# print("Batch Size:", grid_result.best_params_['batch_size'])
# print("Best AUC:", grid_result.best_score_)


In [ ]:
X = data_df.drop(columns=['Activity'])
y = data_df['Activity'].map({'Inactive': 0, 'Active': 1})

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

epochs = 50
batch_size = 64

best_model = Sequential([
    Dense(881, activation='relu', input_shape=(X.shape[1],)),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

metrics = [tf.keras.metrics.AUC()]

best_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=metrics
)

ml_base = 'Supplementary data S7- BCL2_DNN_Rev1_2024_Feb_21'
model_name = ml_base + '.h5'

print("Model Name:", model_name)
print("Training:", model_name)

checkpoint = ModelCheckpoint(
    model_name,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=0
)

history = best_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[checkpoint],
    verbose=0
)

print("Training complete:", model_name)


In [ ]:
ml_base = 'Supplementary data S7- BCL2_DNN_Rev1_2024_Feb_21'
model_name = ml_base + '.h5'

val_loss, val_auc = best_model.evaluate(X_val, y_val)

print("Validation Loss:", val_loss)
print("Validation AUC:", val_auc)
print("Model Name:", model_name)

best_model.summary()
